# LLM Request Quickstart

LLM 호출 + MongoDB 캐시를 한 번에 처리하는 `llm_request` 사용 가이드입니다.

| 함수 | 설명 |
| --- | --- |
| `llm_request` | LLM 호출 + 캐시 (sync) |
| `llm_request_async` | LLM 호출 + 캐시 (async) |
| `llm_request_stream` | 스트리밍 LLM 호출 + 캐시 (sync) |
| `llm_request_stream_async` | 스트리밍 LLM 호출 + 캐시 (async) |
| `done_check` | 입력을 해시하여 캐시 키 생성 + DB 조회 |

**사전 조건**
- LLM Handler 서비스 실행 중 (`http://192.7.9.45:8100`)
- MongoDB 접근 가능 (`mongodb://192.7.9.45:27017/`)

## 설정

In [1]:
!pip install -e ../llm-handler/sdk
!pip install -e ../mdb-handler/sdk
!pip install -e .

Obtaining file:///home/jovyan/work/packages/llm-handler/sdk
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.5/73.5 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.8/78.8 kB 1.8 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 463.6/463.6 kB 8.3 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 9.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 kB 2.5 MB/s eta 0:00:00
  Building editable for kiwoom-llm-handler (pyproject.toml) ... done
  Created wheel for kiwoom-llm-handler: filename=kiwoom_llm_handler-0.1.0-0.editable-py3-none-any.whl size=2990 sha256=6ac7423f1ef8b52f41c0b5ac39c3fb5117cc65431cf

In [ ]:
import os

os.environ["MONGODB_URI"] = "mongodb://192.7.9.45:27017/"
os.environ["MONGODB_DATABASE"] = "kw_ai_dev"
# os.environ["LLM_HANDLER_URL"] = "http://host.docker.internal:8100"
os.environ["LLM_HANDLER_URL"] = "http://192.7.9.45:8100"

from mdb_handler import MongoDBHandler
MongoDBHandler.reset()

import llm_handler
llm_handler.configure()

import llm_handler.aio
llm_handler.aio.configure()

print("설정 완료")

설정 완료


> **⚠️ 데이터베이스 안내**
>
> 이 Quickstart에서는 **`kw_ai_dev`** 데이터베이스에 저장됩니다 (개발/테스트용).
> 실제 에이전트 구축 시에는 **`kw_ai`** 로 변경하여 저장해야 합니다.
>
> ```python
> # 개발/테스트
> os.environ["MONGODB_DATABASE"] = "kw_ai_dev"
>
> # 운영/에이전트
> os.environ["MONGODB_DATABASE"] = "kw_ai"
> ```

---
## 1. `llm_request` — 단건 LLM 호출 + 캐시 (sync)

`llm_request`는 첫 인자 `llm_input`(dict)과 선택 키워드 인자(`cache_mode`, `save_mode`, …)로 LLM 호출, 캐시 조회, DB 저장을 처리합니다.

### 호출 형태

```python
response, meta = llm_request(
    {                       # llm_input: llm_handler.call()에 전달할 인자
        "contents": "질문", # 필수. str 또는 [str | Part | FileRef] 혼합 리스트
        "model": "gemini-3-flash-preview",
        "system": "시스템 프롬프트",   # 선택
        "temperature": 0,             # 선택
        "max_tokens": 200,            # 선택
        "service_name": "my_service", # 선택 (메타에 기록)
    },
    cache_mode=True,         # 캐시 조회 여부 (기본 True)
    save_mode=True,          # DB 저장 여부 (기본 True)
    ttl_seconds=None,        # 캐시 만료 (선택)
    debug=False,             # route_reason (선택)
)
```


### 1-1. 기본 호출 (캐시 + 저장 모두 켬)

In [4]:
from llm_request import llm_request

llm_input = {
    "contents": "대한민국의 수도는?",
    "model": "gemini-3-flash-preview",
    "temperature": 0,
    "max_tokens": 200,
}

response, meta = llm_request(llm_input)  # sync — await 불필요
print(response.text)
print(f"source: {meta['source']}")  # 'live' 또는 'cache'


대한민국의 수도는 **서울**입니다.
source: live


### 1-2. 동일 입력 재호출 → 캐시 히트

In [5]:
response2, meta2 = llm_request(llm_input)

print(f"source: {meta2['source']}")       # 'cache'
print(f"cache_hit: {meta2['cache_hit']}")  # True
print(f"text: {response2.text}")


source: cache
cache_hit: True
text: 대한민국의 수도는 **서울**입니다.


### 1-3. 캐시 없이 LLM만 호출

In [6]:
llm_input_no_cache = {
    "contents": "1+1+1은?",
    "model": "gemini-3-flash-preview",
    "temperature": 0,
    "max_tokens": 200,
}

response3, meta3 = llm_request(
    llm_input_no_cache,
    cache_mode=False,
    save_mode=True,
)
print(response3.text)
print(f"cache_key: {meta3['cache_key']}")


1+1+1은 **3**입니다.
cache_key: d49704e4025d370566983a821ae6d0741511af9eae88283209ce5c4cb1973da2


### 1-4. 시스템 프롬프트 + service_name 지정

In [7]:
llm_input_system = {
    "contents": "ESG 평가 기준을 알려줘",
    "model": "gemini-3-flash-preview",
    "system": "키움증권 ESG 전문가입니다. 간결하게 답변하세요.",
    "service_name": "esg_assistant",
    "max_tokens": 500,
}

response4, meta4 = llm_request(llm_input_system)
print(response4.text)
print(f"meta: {meta4['meta']}")  # service_name, model 기록됨


키움증권 ESG 전문가입니다. ESG 평가의 핵심 기준은 3가지 부문
meta: {'service_name': 'esg_assistant', 'model': 'gemini-3-flash-preview'}


### 1-5. 반환값 구조

모든 `llm_request` 계열 함수는 `(LLMResponse, meta_dict)` 튜플을 반환합니다.

```python
response, meta = llm_request(llm_input)

# response: LLMResponse
print(response.text)          # 모델 응답 텍스트
print(response.model)         # 사용된 모델명
print(response.finish_reason) # "STOP" | "MAX_TOKENS" | "SAFETY" 등
if response.usage:
    u = response.usage
    print(u.input_tokens, u.output_tokens, u.total_tokens)
    print(u.thinking_tokens)  # thinking 모드 사용 시에만 존재

# meta: dict
{
    "source":     "live" | "cache",
    "cache_hit":  bool,
    "cache_key":  str | None,   # SHA-256 해시 (64자)
    "meta":       dict,         # {service_name, model, provider}
    "collection": str | None,   # 저장된 MongoDB 컬렉션명
    # debug=True 시 추가
    "route_reason": str,        # "cache_hit" | "cache_miss" | "cache_off_save_off" 등
}
```


---
## 2. `llm_request_async` — 단건 LLM 호출 + 캐시 (async)

`llm_request`와 동일한 기능의 async 버전입니다. `await`로 호출합니다.
복합 파이프라인(LLM 여러 번 순차 호출)에서 `await`로 이벤트 루프에 양보하여 다른 작업과 병렬 실행할 때 사용합니다.

In [8]:
from llm_request import llm_request_async

response_async, meta_async = await llm_request_async(
    {
        "contents": "파이썬의 장점을 한 줄로 알려줘",
        "model": "gemini-3-flash-preview",
        "temperature": 0,
        "max_tokens": 100,
    },
)
print(response_async.text)
print(f"source: {meta_async['source']}")


파
source: live


---
## 3. 파일·이미지 포함 — multipart `contents`

`contents`에 텍스트와 파일을 **리스트**로 묶어 멀티파트 입력을 전달합니다.

| 방법 | 함수 | 적합한 경우 |
| --- | --- | --- |
| **Files API 업로드** | `llm_handler.aio.upload_file()` → `FileRef` | 대용량 PDF·동영상, 동일 파일 재사용 |
| **인라인 전달** | `Part.from_file()` / `Part.from_bytes()` | 소용량 이미지·문서, 1회성 |
| **URI 참조** | `Part.from_uri()` | 이미 업로드된 URI 또는 공개 URL |

이 노트북과 같은 폴더에 **`quickstart.pdf`** 샘플을 두었습니다. 아래 셀에서 **방법 1** `Part.from_file()` 과 **방법 2** `llm_handler.aio.upload_file()`(Files API) 을 연속으로 실행합니다.

In [9]:
from pathlib import Path

import llm_handler.aio
from llm_handler import Part
from llm_request import llm_request_async

# quickstart.ipynb와 같은 폴더의 quickstart.pdf
for _p in (Path("quickstart.pdf"), Path("packages/llm-request/quickstart.pdf")):
    if _p.is_file():
        SAMPLE_PDF = _p.resolve()
        break
else:
    raise FileNotFoundError(
        "quickstart.pdf를 찾을 수 없습니다. "
        f"노트북과 같은 디렉터리에 두거나, 커널 cwd를 확인하세요. cwd={Path.cwd()}"
    )

# ── 방법 1: 인라인 전달 — Part.from_file() ───────────────────────────
response_pdf, meta_pdf = await llm_request_async(
    {
        "contents": [
            "첨부 PDF의 목차나 제목 수준에서 핵심 주제를 3줄로 요약해줘.",
            Part.from_file(str(SAMPLE_PDF)),
        ],
        "model": "gemini-3-flash-preview",
        "max_tokens": 1400,
        "service_name": "quickstart_inline_pdf",
    },
    cache_mode=True,
    save_mode=False,
)
print("=== 방법 1 Part.from_file ===")
print(response_pdf.text)
print(f"source: {meta_pdf['source']}")
print()

# ── 방법 2: Files API — Handler에 업로드 후 FileRef를 contents에 포함 ─
file_ref = await llm_handler.aio.upload_file(
    str(SAMPLE_PDF),
    display_name="quickstart_sample",
)
print(f"업로드 완료: uri={file_ref.uri!r}")

response_upload, meta_upload = await llm_request_async(
    {
        "contents": [
            "업로드된 PDF를 한 문단(3~4문장)으로 요약해줘. 수치가 있으면 하나만 골라 언급해줘.",
            file_ref,
        ],
        "model": "gemini-3-flash-preview",
        "max_tokens": 1350,
        "service_name": "quickstart_files_api",
    },
    cache_mode=True,
    save_mode=True,
)
print("=== 방법 2 Files API (upload_file) ===")
print(response_upload.text)
print(f"source: {meta_upload['source']}")

# ── 방법 3: URI — Part.from_uri("https://...", "application/pdf") ───
# (공개 URL이나 이미 업로드된 files/... URI가 있을 때)
# 방법2와 llm_handler.aio.upload_file과정 제외 입력방식 동일


=== 방법 1 Part.from_file ===
첨부된 '2024 KB금융그룹 지속가능경영보고서(투자자 관점)'의 목차와 주요 제목을 바탕으로 요약한 핵심 주제 3줄은 다음과 같습니다.

1. **5대 중대 이슈 중심의 공시**: IFRS 지속가능성 공시기준(S1·S2)에 따라 투자자 의사결정에 핵심적인 5대 이슈(리스크 관리, 기후위기 대응, 포용금융, 금융소비자 보호, 디지털 혁신 및 기술)를 선정하고 영역별 거버넌스, 전략, 위험관리, 지표를 상세히 기술합니다.
2. **기후변화 대응 및 사회적 책임 강화**: 'KB Net Zero S.T.A.R.' 등 넷제로 달성을 위한 기후 전략과 시나리오 분석 결과를 공개하고, 취약계층 지원을 위한 포용금융 및 소비자 권익 보호를 통한 사회적 가치 창출에 집중합니다.
3. **디지털 혁신과 체계적 리스크 관리**: AI 및 마이데이터 등 신기술 기반의 디지털 혁신으로 미래 성장 동력을 확보하는 동시에, 이사회 중심의 통합 리스크 관리 체계를 구축하여 그룹의 재무적·비재무적 지속가능성을 도모합니다.
source: live

업로드 완료: uri='https://generativelanguage.googleapis.com/v1beta/files/7n0z8it1e3pb'
=== 방법 2 Files API (upload_file) ===
2024 KB금융그룹 지속가능경영보고서(투자자 관점)는 리스크 관리, 기후위기 대응, 포용금융 등 5대 중대 이슈를 중심으로 그룹의 ESG 전략과 재무적 영향을 상세히 기술하고 있습니다. 이 보고서는 IFRS 지속가능성 공시기준(S1·S2)을 선제적으로 적용하여 투자자 의사결정에 필요한 정량적·정성적 정보를 체계적으로 공개한 것이 특징입니다. 특히 'KB Net Zero S.T.A.R.' 전략을 통한 탄소 중립 추진과 AI 거버넌스 체계 구축 등 지속가능한 성장 동력 확보를 위한 구체적인 로드맵을 제시하고 있습니다. 디지털 혁신의 주요 성과로 그룹의 대표 앱인 KB스타뱅킹의 월간 활성 

---
## 4. `llm_request_stream` — 스트리밍 LLM 호출 (sync)

LLM 응답을 chunk 단위로 실시간 수신합니다. 사용자에게 텍스트가 점진적으로 나타나는 UX에 적합합니다.

항상 `(Iterator[StreamChunk], meta)` 튜플을 반환합니다. 캐시 히트든 미스든 동일한 방식으로 chunk를 순회합니다.

In [10]:
from llm_request import llm_request_stream

llm_input_stream = {
        "contents": (
            "다음 주제에 대해 3000자 이상의 상세한 보고서를 작성해줘.\n\n"
            "주제: 인공지능(AI)의 발전 역사와 미래 전망\n\n"
            "1. 1950년대~1970년대 AI 태동기 (튜링 테스트, 퍼셉트론, 첫 번째 AI 겨울)\n"
            "2. 1980년대~1990년대 전문가 시스템과 두 번째 AI 겨울\n"
            "3. 2000년대~2010년대 머신러닝 부활, 딥러닝 혁명 (ImageNet, AlphaGo)\n"
            "4. 2020년대 대규모 언어모델(GPT, Gemini 등)과 생성형 AI\n"
            "5. 향후 10년간 AI가 금융, 의료, 교육 분야에 미칠 영향 예측\n\n"
            "각 시대별로 핵심 인물, 주요 기술, 한계점, 사회적 반응을 포함해줘."
        ),
        "model": "gemini-2.5-flash",
        "temperature": 0.7,
        "max_tokens": 4000,
}

stream, meta = llm_request_stream(
    llm_input_stream,
    cache_mode=False,
    save_mode=True,
)
chunk_count = 0
total_text = []
for chunk in stream:
    total_text.append(chunk.text or "")
    chunk_count += 1
    if chunk_count % 5 == 0:
        t = chunk.text or ""
        print(f"  [chunk #{chunk_count:3d}] ...{t[:60]!r}", flush=True)
print(f"\n총 {chunk_count}개 chunk 수신, {''.join(total_text).__len__()}자")
print(f"cache_hit: {meta['cache_hit']}, source: {meta['source']}")


  [chunk #  5] ...' 시대별 핵심 인물, 주요 기술, 한계점, 그리고 사회적 반응을 심층적으로 분석합니다. AI의 흥망성쇠를 이'
  [chunk # 10] ...' 구별할 수 없는 대화를 나눌 수 있다면 지능이 있다고 볼 수 있다고 주장했습니다. 이는 AI 연구의 궁극적'
  [chunk # 15] ...' 형태의 인공신경망 연구를 이끌었습니다. 퍼셉트론은 입력값을 받아 가중치를 곱하고 합산하여 특정 임계값을 넘'
  [chunk # 20] ...'퍼셉트론(Perceptrons)"이라는 책은 이러한 한계를 지적하며 신경망 연구에 대한 회의론을 확산시키는 '
  [chunk # 25] ...'방부(DARPA)와 같은 주요 연구 자금 지원 기관의 AI 예산 삭감으로 이어졌고, AI 연구는 침체기에 접'
  [chunk # 30] ..."액 감염 진단 시스템인 '마이신(MYCIN)'과 DEC(Digital Equipment Corporation"
  [chunk # 35] ...' 방법을 제시하며 신경망 연구에 새로운 활력을 불어넣었습니다. 이는 초기 퍼셉트론의 한계를 극복하고 비선형 '
  [chunk # 40] ...' 것은 어려웠고,'

총 40개 chunk 수신, 3734자
cache_hit: False, source: live


---
## 5. `llm_request_stream_async` — 스트리밍 LLM 호출 (async)

`llm_request_stream`의 async 버전입니다. `await`로 호출하면 `(AsyncIterator[StreamChunk], meta)` 튜플을 반환하고 `async for`로 chunk를 순회합니다.

In [11]:
from llm_request import llm_request_stream_async

llm_input_async_stream = {
        "contents": (
            "다음 주제에 대해 3000자 이상의 상세한 기술 분석 보고서를 작성해줘.\n\n"
            "주제: 블록체인 기술의 진화와 산업별 적용 현황\n\n"
            "1. 블록체인 1.0 — 비트코인과 분산 원장의 탄생 (합의 알고리즘, 작업증명)\n"
            "2. 블록체인 2.0 — 이더리움과 스마트 컨트랙트 혁명 (DeFi, NFT)\n"
            "3. 블록체인 3.0 — 확장성 솔루션 (Layer 2, 샤딩, 지분증명 전환)\n"
            "4. 금융 분야: CBDC, 토큰증권(STO), 크로스보더 결제 사례\n"
            "5. 물류·공급망: 원산지 추적, IoT 연계, 위변조 방지\n"
            "6. 의료: 전자건강기록(EHR) 공유, 임상시험 데이터 무결성\n"
            "7. 한계점과 향후 과제 (에너지 문제, 규제, 프라이버시)\n\n"
            "각 섹션마다 구체적인 프로젝트/기업 사례를 2개 이상 포함해줘."
        ),
        "model": "gemini-2.5-flash",
        "temperature": 0.7,
        "max_tokens": 4000,
}

stream, meta = await llm_request_stream_async(
    llm_input_async_stream,
    cache_mode=False,
    save_mode=True,
)
chunk_count = 0
total_text = []
async for chunk in stream:
    total_text.append(chunk.text or "")
    chunk_count += 1
    if chunk_count % 5 == 0:
        t = chunk.text or ""
        print(f"  [chunk #{chunk_count:3d}] ...{t[:60]!r}", flush=True)
print(f"\n총 {chunk_count}개 chunk 수신, {''.join(total_text).__len__()}자")
print(f"cache_hit: {meta['cache_hit']}, source: {meta['source']}")


  [chunk #  5] ...'체인 기술의 태동부터 현재에 이르기까지의 진화 과정을 단계별로 분석하고, 주요 산업 분야에서의 실제 적용 현'
  [chunk # 10] ...' 네트워크 참여자들에게 분산되어 저장되고 공유됩니다. 특정 중앙 서버에 의존하지 않으므로 단일 실패 지점(S'
  [chunk # 15] ...' 이 과정은 막대한 컴퓨팅 자원과 에너지를 요구하므로, 악의적인 공격자가 네트워크를 장악하기 위해서는 전체 '
  [chunk # 20] ...'을 사용하지만, 블록 생성 시간을 2.5분으로 단축하고 해싱 알고리즘으로 Scrypt를 채택하여 비트코인보다'
  [chunk # 25] ...'스마트 컨트랙트(Smart Contract):** 미리 정의된 조건이 충족되면 자동으로 실행되는 계약 코드를'
  [chunk # 30] ...'큰(NFT)의 표준입니다.\n\n**혁신 분야:**\n\n1.  **탈중앙화 금융 (DeFi, Decentraliz'

총 33개 chunk 수신, 3167자
cache_hit: False, source: live


---
## 7. `done_check` — 캐시 키 생성 + DB 조회

`llm_input`을 정규화 → SHA-256 해시하여 캐시 키를 생성하고, MongoDB에 이미 처리된 결과가 있는지 조회합니다.

`llm_request` 내부에서도 동일한 흐름으로 동작합니다.

In [12]:
from llm_request import done_check

llm_input_done = {
    "contents": "대한민국의 수도는?",
    "model": "gemini-3-flash-preview",
    "temperature": 0,
    "max_tokens": 200,
}

# ① 캐시 키만 생성 (DB 조회 없이)
dc = done_check(llm_input_done, lookup_cache=False)
print(f"cache_key:    {dc.cache_key}")    # SHA-256 (64자)
print(f"canonical:    {dc.canonical_json[:60]}...")  # 정규화된 입력
print(f"collection:   {dc.collection}")
print(f"cached:       {dc.cached}")      # None (조회 안 했으므로)

print()

# ② 캐시 조회까지 수행
dc2 = done_check(llm_input_done, lookup_cache=True)
if dc2.cached:
    print(f"캐시 히트! text: {dc2.cached['result']['text'][:60]}")
else:
    print("캐시 미스 — 저장된 결과 없음")
print(f"cache_error:  {dc2.cache_error}")  # DB 연결 오류 시 True


cache_key:    c569f6df28ac345d39cf49410cc677b30ff8bf799495e1c18f5de71dae48b7bf
canonical:    {"contents":"대한민국의 수도는?","max_tokens":200,"model":"gemini-3-...
collection:   agent_llm
cached:       None

캐시 히트! text: 대한민국의 수도는 **서울**입니다.
cache_error:  False


### 캐시 키 동작 원리

`llm_input`에서 캐시에 영향을 주는 필드만 추출 → 키 정렬·None→빈문자열 정규화 → SHA-256 해시

**캐시 키에 포함되는 필드:**
`contents`, `system`, `history`, `model`, `max_tokens`, `temperature`, `cached_content`, `service_name`, `seed`, `output`, `schema`, `thinking`, `tools`, `tool_config`, `media_resolution`

이 중 하나라도 다르면 **다른 캐시 키**가 됩니다.


### 7-4. 반환값 구조 — `DoneCheckResult`

```python
@dataclass(frozen=True)
class DoneCheckResult:
    collection: str              # 저장 대상 MongoDB 컬렉션명
    cache_key: str               # SHA-256 해시 (64자)
    canonical_json: str          # 정규화된 입력 JSON 문자열
    cached: dict | None          # 캐시 히트 시 저장된 문서, 미스 시 None
    cache_error: bool            # DB 조회 에러 발생 여부
```

---
# Appendix: 에이전트 개발 패턴

`llm_request`를 사용하는 에이전트 개발 시 자주 쓰는 패턴입니다.

## 패턴 A: 단건 처리 — `llm_request_async`

단건 LLM 호출 + 캐시. 에이전트 내 단일 LLM 호출의 기본 패턴입니다.

In [13]:
from llm_request import llm_request_async

response_a, meta_a = await llm_request_async(
    {
        "contents": "삼성전자 반도체 사업 현황을 두 줄로 요약해줘.",
        "model": "gemini-3-flash-preview",
        "temperature": 0,
        "max_tokens": 150,
        "service_name": "fund_comment",
    },
    cache_mode=True,
    save_mode=True,
)

print(response_a.text)
print(f"source:    {meta_a['source']}")       # 'live' 또는 'cache'
print(f"cache_hit: {meta_a['cache_hit']}")
if response_a.usage:
    u = response_a.usage
    print(f"tokens:    total={u.total_tokens} (in={u.input_tokens}, out={u.output_tokens})")


삼성전자의
source:    live
cache_hit: False
tokens:    total=164 (in=18, out=3)


---
## 패턴 B: N건 병렬 — `llm_request_async` + 타이밍 로그

스트리밍 대신 **응답 한 덩어리**로 받고, **시간 로그 + 요약 표**로 비동기가 잘 도는지 보기 쉽게 했습니다.

- 펀드마다 **DB 조회 지연**을 다르게 두어 (`0.4s × 인덱스`) LLM 요청 시각이 어긋납니다.
- 프롬프트에 **짧은 단락 요청 + `max_tokens` 여유**를 두어 LLM이 조금 더 걸리게 했습니다.
- 로그에 `[경과초] 코드 DB완료` → `LLM시작` → `LLM끝(소요초)` 가 찍혀, **겹쳐 진행**되는지 눈으로 확인할 수 있습니다.

```
005930: task → DB(즉시) → [LLM 2~4s] ─────────────┐
000660: task → DB(0.4s) → [LLM ...]              ├─ as_completed
035420: task → DB(0.8s) → [LLM ...]              │
068270: task → DB(1.2s) → [LLM ...]              ┘
```

In [14]:
import asyncio
import time
from llm_request import llm_request_async

FUNDS = [
    {"code": "005930", "name": "삼성전자",  "sector": "반도체"},
    {"code": "000660", "name": "SK하이닉스", "sector": "반도체"},
    {"code": "035420", "name": "NAVER",      "sector": "IT"},
    {"code": "068270", "name": "셀트리온",   "sector": "바이오"},
]

# 펀드별 DB 조회 지연(초) — LLM 요청 시각을 어긋나게 해 비동기 로그가 겹쳐 보이게 함
DB_STAGGER_SEC = 0.42


async def process_fund(fund: dict, idx: int, wall0: float) -> dict:
    """DB 조회(시뮬) 후 LLM 호출. wall0 기준 경과 시간 로그."""
    code = fund["code"]

    def rel() -> float:
        return time.perf_counter() - wall0

    print(f"[{rel():5.2f}s] {code} task 시작")

    # 실제 에이전트: fund_data = await db.fetch(fund["code"])
    await asyncio.sleep(DB_STAGGER_SEC * idx)
    t_db = rel()
    print(f"[{t_db:5.2f}s] {code} DB 조회 완료")

    fund_data = f"{fund['name']}({fund['sector']})"

    t_in = rel()
    print(f"[{t_in:5.2f}s] {code} LLM 호출 시작")

    t_llm_wall = time.perf_counter()
    response, meta = await llm_request_async(
        {
            "contents": (
                f"{fund_data} 펀드에 대해 투자 관점에서 "
                f"2~3문장으로 코멘트를 작성해줘. "
                f"리스크와 기회를 각각 한 문장씩 포함해줘."
            ),
            "model": "gemini-3-flash-preview",
            "temperature": 0.4,
            "max_tokens": 1350,
            "service_name": "fund_comment",
        },
        cache_mode=True,
        save_mode=True,
    )
    llm_sec = time.perf_counter() - t_llm_wall
    t_out = rel()
    print(
        f"[{t_out:5.2f}s] {code} LLM 응답 수신 "
        f"(순수 LLM {llm_sec:.2f}s, {len(response.text or '')}자, source={meta['source']})",
        f"         {code} >> {(response.text or '')[:80]}{'…' if len(response.text or '') > 80 else ''}"
    )

    return {
        "code": code,
        "text": response.text,
        "source": meta["source"],
        "cache_key": meta["cache_key"],
        "t_db_done": t_db,
        "t_llm_start": t_in,
        "t_llm_end": t_out,
        "llm_sec": llm_sec,
    }


print("=" * 72)
print(f"{len(FUNDS)}개 펀드 병렬 (DB 지연 {DB_STAGGER_SEC}s×인덱스, max_tokens=1350)")
print("타임스탬프는 아래 '패턴 B 시작' 시각 기준 경과 초입니다.")
print("=" * 72)

wall0 = time.perf_counter()
tasks = [asyncio.create_task(process_fund(f, i, wall0)) for i, f in enumerate(FUNDS)]
many_results = []

for coro in asyncio.as_completed(tasks): 
    result = await coro
    wall = time.perf_counter() - wall0
    many_results.append(result)
    print(f"  >> as_completed [{wall:5.2f}s] {result['code']} 전체 Task 종료")

print("=" * 72)
print(f"벽시계 전체: {time.perf_counter() - wall0:.2f}s")



4개 펀드 병렬 (DB 지연 0.42s×인덱스, max_tokens=1350)
타임스탬프는 아래 '패턴 B 시작' 시각 기준 경과 초입니다.
[ 0.00s] 005930 task 시작
[ 0.00s] 000660 task 시작
[ 0.00s] 035420 task 시작
[ 0.00s] 068270 task 시작
[ 0.00s] 005930 DB 조회 완료
[ 0.00s] 005930 LLM 호출 시작
[ 0.42s] 000660 DB 조회 완료
[ 0.42s] 000660 LLM 호출 시작
[ 0.84s] 035420 DB 조회 완료
[ 0.84s] 035420 LLM 호출 시작
[ 1.30s] 068270 DB 조회 완료
[ 1.30s] 068270 LLM 호출 시작
[ 4.70s] 000660 LLM 응답 수신 (순수 LLM 4.28s, 171자, source=live)          000660 >> SK하이닉스는 AI 산업 성장의 핵심인 HBM(고대역폭메모리) 분야에서 독보적인 기술 경쟁력을 확보하고 있어 장기적인 수익성 개선이 기대되는 매…
  >> as_completed [ 4.70s] 000660 전체 Task 종료
[ 5.39s] 005930 LLM 응답 수신 (순수 LLM 5.39s, 252자, source=live)          005930 >> 삼성전자는 AI 산업 성장에 따른 고대역폭메모리(HBM) 수요 확대와 메모리 업황 회복의 직접적인 수혜가 기대되어 중장기적 성장 잠재력이 매우 …
  >> as_completed [ 5.39s] 005930 전체 Task 종료
[ 5.87s] 035420 LLM 응답 수신 (순수 LLM 5.03s, 190자, source=live)          035420 >> 네이버(IT) 펀드는 생성형 AI '하이퍼클로바X'를 기반으로 한 검색·커머스의 수익 모델 고도화와 글로벌 콘텐츠 시장 확장을 통해 강력한 재도…
  >> as_completed [ 5.88s] 035420 전체 Task 종료
[ 6

### 결과 확인 — 요약 표

같은 **패턴 B 시작 시각** 기준으로 DB 완료·LLM 구간·순수 LLM 소요시간을 한눈에 봅니다.  
`LLM시작` 시각이 DB끝 직후에 연달아 찍히면 비동기 병렬 진행이 확인됩니다. 동시 호출 제한은 llm-handler(`gemini_max_concurrency`)가 담당합니다.
결과를 통해 앞선 과정(예시 DB)이 종료되는 즉시 llm_request_async를 실행하고, 응답을 바로 받을 수 있습니다.

In [15]:
rows = sorted(many_results, key=lambda x: x["code"])
hdr = f"{'코드':<8} {'DB끝(s)':>8} {'LLM시작(s)':>10} {'LLM끝(s)':>9} {'순수LLM(s)':>10} {'source':<8}"
print(hdr)
print("-" * len(hdr))
for r in rows:
    print(
        f"{r['code']:<8} "
        f"{r['t_db_done']:8.2f} "
        f"{r['t_llm_start']:10.2f} "
        f"{r['t_llm_end']:9.2f} "
        f"{r['llm_sec']:10.2f} "
        f"{r['source']:<8}"
    )

print()
for r in rows:
    preview = r["text"].replace("\n", " ")[:100]
    print(f"[{r['code']}] {preview}{'…' if len(r['text']) > 100 else ''}")

코드         DB끝(s)   LLM시작(s)   LLM끝(s)   순수LLM(s) source  
----------------------------------------------------------
000660       0.42       0.42      4.70       4.28 live    
005930       0.00       0.00      5.39       5.39 live    
035420       0.84       0.84      5.87       5.03 live    
068270       1.30       1.30      6.24       4.94 live    

[000660] SK하이닉스는 AI 산업 성장의 핵심인 HBM(고대역폭메모리) 분야에서 독보적인 기술 경쟁력을 확보하고 있어 장기적인 수익성 개선이 기대되는 매력적인 투자처입니다. 다만, 메모리 …
[005930] 삼성전자는 AI 산업 성장에 따른 고대역폭메모리(HBM) 수요 확대와 메모리 업황 회복의 직접적인 수혜가 기대되어 중장기적 성장 잠재력이 매우 높습니다. 다만, HBM 시장 내 주…
[035420] 네이버(IT) 펀드는 생성형 AI '하이퍼클로바X'를 기반으로 한 검색·커머스의 수익 모델 고도화와 글로벌 콘텐츠 시장 확장을 통해 강력한 재도약의 기회를 맞이하고 있습니다. 다만…
[068270] 셀트리온 펀드는 짐펜트라의 미국 시장 안착과 신약 개발 기업으로의 도약을 통해 강력한 중장기 성장 모멘텀을 보유하고 있습니다. 다만, 글로벌 바이오시밀러 시장의 가격 경쟁 심화와 …


---
## 정리

테스트 데이터를 삭제합니다.

In [16]:
from llm_request import done_check

mdb = MongoDBHandler()
col = mdb.get_database("kw_ai_dev", "agent_llm")
deleted = 0

# 섹션 1 기본 예제
for llm_in in [llm_input, llm_input_no_cache, llm_input_system]:
    dc = done_check(llm_in, lookup_cache=False)
    res = col.delete_one({"_id": dc.cache_key})
    deleted += res

# 섹션 2 async 예제
if meta_async.get("cache_key"):
    res = col.delete_one({"_id": meta_async["cache_key"]})
    deleted += res

# 섹션 3 PDF (Part.from_file + Files API)
if meta_pdf.get("cache_key"):
    res = col.delete_one({"_id": meta_pdf["cache_key"]})
    deleted += res
if meta_upload.get("cache_key"):
    res = col.delete_one({"_id": meta_upload["cache_key"]})
    deleted += res

# 섹션 4, 5 stream 예제
for llm_in in [llm_input_stream, llm_input_async_stream]:
    dc = done_check(llm_in, lookup_cache=False)
    res = col.delete_one({"_id": dc.cache_key})
    deleted += res

# Appendix 패턴 A
if meta_a.get("cache_key"):
    res = col.delete_one({"_id": meta_a["cache_key"]})
    deleted += res

# Appendix 패턴 B (cache_key는 process_fund 결과에서 직접 참조)
for r in many_results:
    if r.get("cache_key"):
        res = col.delete_one({"_id": r["cache_key"]})
        deleted += res

print(f"정리 완료: {deleted}건 삭제")


정리 완료: 12건 삭제
